Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

Tracking agent behavior with logging, analytics, and debugging.
Transforming prompts, tool selection, and output formatting.
Adding retries, fallbacks, and early termination logic.
Applying rate limits, guardrails, and PII detection.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model=init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000187D1800830>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000187D1801550>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

Summarization MiddleWare

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

Long-running conversations that exceed context windows.

Multi-turn dialogues with extensive history.

Applications where preserving full conversation context matters.

In [2]:
from langgraph import checkpoint
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage

##messagebased summarization
agent=create_agent(
    model="groq:qwen/qwen3-32b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [4]:
#run the thread id
config={"configurable":{"thread_id":"test-1"}}

In [5]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='70281cca-359d-4e87-8a4b-2b3302a4d46a'), AIMessage(content="<think>\nOkay, so I need to figure out what 2 + 2 is. Let me start by recalling basic addition. When you add two numbers together, you're combining their quantities. So, 2 is a number that represents two units. If I have two apples and then get two more apples, how many apples do I have in total? Let me visualize this. First, there are two apples. Then, I add another two apples. So, 2 plus 2. That should be four apples, right? \n\nWait, maybe I should break it down step by step. Let's think about counting. If I start at 2 and then count two more: 2, 3, 4. So that's two steps. So starting from 2, adding 1 gets me to 3, and adding another 1 gets me to 4. So that's two added together. Hmm, that seems straightforward. \n\nAnother way to look at it is using fingers. If I hold up two fingers on one hand and two on the other, h